# Wizualizacje interaktywne - Plotly i Dash

## Interaktywność zmienia wszystko!

W tym notebooku nauczysz się tworzyć **interaktywne** wykresy, które użytkownik może eksplorować: zoomować, klikać, filtrować!

**Matplotlib i Seaborn** = statyczne obrazki  
**Plotly** = interaktywne wykresy w przeglądarce  
**Dash** = pełne dashboardy webowe  

## 1. Wprowadzenie do Plotly

**Plotly** to biblioteka do tworzenia interaktywnych wizualizacji.

### Dwa sposoby pracy:

1. **Plotly Express** - szybkie, proste wykresy (jak Seaborn)
2. **Plotly Graph Objects** - pełna kontrola (jak Matplotlib)

### Zalety Plotly:

✅ **Interaktywność** - zoom, pan, hover, filtrowanie  
✅ **Piękne wykresy** - profesjonalnie wyglądają od razu  
✅ **Wykresy 3D** - lepsze niż Matplotlib  
✅ **Eksport** - do HTML, można otworzyć w przeglądarce  
✅ **Dashboardy** - Dash Framework do tworzenia aplikacji webowych  

### Instalacja:

```bash
pip install plotly
pip install dash  # dla dashboardów
```

In [1]:
!pip install plotly


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install dash


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### Setup

In [3]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ⚠️ WAŻNE: Konfiguracja wyświetlania w Jupyter Notebook
import plotly.io as pio
# pio.renderers.default = 'notebook'  # Dla Jupyter Notebook
pio.renderers.default = 'jupyterlab'  # Dla JupyterLab
# pio.renderers.default = 'iframe'  # Jeśli 'notebook' nie działa

print("Plotly gotowy!")
print(f"Renderer: {pio.renderers.default}")
print(f"Wersja Plotly: {px.__version__ if hasattr(px, '__version__') else 'unknown'}")

Plotly gotowy!
Renderer: jupyterlab
Wersja Plotly: unknown


### ⚠️ Problem: Wykresy się nie wyświetlają?

Jeśli po wykonaniu `fig.show()` **nic się nie wyświetla**, wypróbuj te rozwiązania:

**Rozwiązanie 1: Zmień renderer** (już dodane w komórce powyżej)
```python
import plotly.io as pio
pio.renderers.default = 'notebook'  # Spróbuj tego
# pio.renderers.default = 'jupyterlab'  # Albo tego
# pio.renderers.default = 'iframe'  # Albo tego
```

**Rozwiązanie 2: Zainstaluj brakujące biblioteki**
```bash
pip install nbformat ipywidgets
jupyter nbextension enable --py widgetsnbextension
```

**Rozwiązanie 3: Restart kernela**
- Kernel → Restart & Run All

**Rozwiązanie 4: Użyj zewnętrznej przeglądarki**
```python
fig.show(renderer='browser')  # Otworzy w przeglądarce
```

**Rozwiązanie 5: Zapisz do HTML i otwórz**
```python
fig.write_html('wykres.html')  # Zapisz do pliku
# Potem otwórz plik wykres.html w przeglądarce
```

**Które środowisko używasz?**
- **Jupyter Notebook** → `pio.renderers.default = 'notebook'`
- **JupyterLab** → `pio.renderers.default = 'jupyterlab'`
- **VS Code** → `pio.renderers.default = 'notebook'` lub `'iframe'`
- **Google Colab** → Działa od razu, nie trzeba konfigurować

---

In [4]:
!pwd

/home/jgrynczewski/2/home/jgrynczewski/PycharmProjects/wiz-dev/04_plotly


In [5]:
# Wczytaj dane
df_prac = pd.read_csv('data/pracownicy.csv')
df_pogoda = pd.read_csv('data/pogoda.csv')
df_pogoda['data'] = pd.to_datetime(df_pogoda['data'])
df_sprzedaz = pd.read_csv('data/sprzedaz.csv')
df_sprzedaz['data'] = pd.to_datetime(df_sprzedaz['data'])
df_sprzedaz['wartosc_sprzedazy'] = df_sprzedaz['kwota'] * df_sprzedaz['ilosc']

FileNotFoundError: [Errno 2] No such file or directory: 'data/pracownicy.csv'

## 2. Plotly Express - szybkie wykresy

Plotly Express to wysokopoziomowe API - podobne do Seaborn, ale interaktywne!

### 2.1 Scatter plot

In [ ]:
# Prosty scatter plot
fig = px.scatter(df_prac, 
                 x='wiek', 
                 y='pensja',
                 title='Wiek vs Pensja (INTERAKTYWNY!)')

fig.show()

print("\n💡 WYPRÓBUJ INTERAKTYWNOŚĆ:")
print("- Najedź myszką na punkt → zobaczysz dokładne wartości")
print("- Zaznacz obszar → zoom")
print("- Dwuklik → powrót do widoku początkowego")
print("- Przyciski w prawym górnym rogu → więcej opcji!")

### Scatter z kolorowaniem i rozmiarem

In [ ]:
# Zaawansowany scatter
fig = px.scatter(df_prac,
                 x='wiek',
                 y='pensja',
                 color='dzial',          # Kolor według działu
                 size='staz',            # Rozmiar według stażu
                 hover_data=['imie'],    # Pokaż imię przy hover
                 title='Wiek vs Pensja (kolor=dział, rozmiar=staż)')

fig.update_layout(height=600)
fig.show()

### 2.2 Line plot

In [ ]:
# Line plot - temperatura w czasie
fig = px.line(df_pogoda,
              x='data',
              y='temperatura',
              title='Temperatura w czasie',
              markers=True)  # Dodaj markery

fig.update_layout(hovermode='x unified')  # Hover pokazuje wszystkie wartości dla danej daty
fig.show()

### Line plot z wieloma liniami

In [ ]:
# Agregacja - sprzedaż według kategorii w czasie
df_sprzedaz['miesiac'] = df_sprzedaz['data'].dt.to_period('M').astype(str)
sprzedaz_miesiac = df_sprzedaz.groupby(['miesiac', 'kategoria'])['wartosc_sprzedazy'].sum().reset_index()

fig = px.line(sprzedaz_miesiac,
              x='miesiac',
              y='wartosc_sprzedazy',
              color='kategoria',          # Osobna linia dla każdej kategorii
              title='Sprzedaż według kategorii',
              markers=True)

fig.update_layout(hovermode='x unified')
fig.show()

print("💡 Kliknij na nazwę kategorii w legendzie → ukryj/pokaż linię!")

### 2.3 Bar plot

In [ ]:
# Bar plot - średnia pensja według działów
avg_pensja = df_prac.groupby('dzial')['pensja'].mean().reset_index()

fig = px.bar(avg_pensja,
             x='dzial',
             y='pensja',
             title='Średnia pensja według działów',
             color='pensja',              # Kolor według wysokości
             color_continuous_scale='Viridis')  # Paleta

fig.show()

### 2.4 Histogram

In [ ]:
# Histogram z grupowaniem
fig = px.histogram(df_prac,
                   x='pensja',
                   color='dzial',           # Osobny kolor dla każdego działu
                   barmode='overlay',       # 'overlay', 'group', 'stack'
                   title='Rozkład pensji według działów',
                   nbins=20)

fig.update_traces(opacity=0.6)
fig.show()

### 2.5 Box plot

In [ ]:
# Box plot interaktywny
fig = px.box(df_prac,
             x='dzial',
             y='pensja',
             color='dzial',
             title='Rozkład pensji według działów (box plot)',
             points='all')  # Pokaż wszystkie punkty!

fig.show()

print("💡 points='all' pokazuje WSZYSTKIE punkty danych - kliknij na nie!")

### 2.6 Violin plot

In [ ]:
# Violin plot
fig = px.violin(df_prac,
                x='dzial',
                y='pensja',
                color='dzial',
                title='Violin plot - pensje według działów',
                box=True,              # Dodaj box plot wewnątrz
                points='all')          # Pokaż punkty

fig.show()

### 2.7 Sunburst - hierarchiczny wykres kołowy

In [ ]:
# Sunburst - sprzedaż według kategorii i produktów
fig = px.sunburst(df_sprzedaz,
                  path=['kategoria', 'produkt'],   # Hierarchia
                  values='wartosc_sprzedazy',
                  title='Hierarchia sprzedaży: Kategoria → Produkt')

fig.show()

print("💡 Kliknij na segment → zoom do podkategorii!")

---
## Ćwiczenie 4.1 (w trakcie) - Plotly Express wykres

**Cel:** Stworzyć interaktywny scatter plot.

**Zadanie:**
1. Użyj danych pracowników
2. Utwórz scatter plot: staż (x) vs pensja (y)
3. Koloruj według działu
4. Ustaw rozmiar według wieku
5. Dodaj hover_data=['imie', 'wiek']
6. Dodaj tytuł
7. Ustaw wysokość na 700px

**Czas:** 10 minut

In [ ]:
# TWÓJ KOD TUTAJ



<details>
<summary><b>🔍 Kliknij, aby zobaczyć rozwiązanie</b></summary>

```python
fig = px.scatter(df_prac,
                 x='staz',
                 y='pensja',
                 color='dzial',
                 size='wiek',
                 hover_data=['imie', 'wiek'],
                 title='Staż vs Pensja (interaktywny)')

fig.update_layout(height=700)
fig.show()

print("Wypróbuj:")
print("1. Najedź na punkt → zobacz imię i wiek")
print("2. Zaznacz obszar → zoom")
print("3. Kliknij dział w legendzie → ukryj/pokaż")
```

</details>

---
## 3. Plotly Graph Objects - pełna kontrola

Graph Objects daje ci pełną kontrolę - jak Matplotlib, ale interaktywne!

### 3.1 Podstawy - Figure i Trace

In [ ]:
# Prosty wykres z Graph Objects
fig = go.Figure()

# Dodaj trace (linię)
fig.add_trace(go.Scatter(
    x=df_pogoda['data'].head(50),
    y=df_pogoda['temperatura'].head(50),
    mode='lines+markers',
    name='Temperatura',
    line=dict(color='red', width=2),
    marker=dict(size=6)
))

# Dostosuj layout
fig.update_layout(
    title='Temperatura - Graph Objects',
    xaxis_title='Data',
    yaxis_title='Temperatura [°C]',
    hovermode='x unified'
)

fig.show()

### 3.2 Wiele trace'ów na jednym wykresie

In [ ]:
# Wykres z wieloma liniami
fig = go.Figure()

dane = df_pogoda.head(100)

# Dodaj trzy linie
fig.add_trace(go.Scatter(
    x=dane['data'],
    y=dane['temperatura'],
    mode='lines',
    name='Temperatura',
    line=dict(color='red', width=2)
))

fig.add_trace(go.Scatter(
    x=dane['data'],
    y=dane['opady']*3,  # Skaluj dla widoczności
    mode='lines',
    name='Opady × 3',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=dane['data'],
    y=dane['wilgotnosc']/2,
    mode='lines',
    name='Wilgotność / 2',
    line=dict(color='green', width=2)
))

fig.update_layout(
    title='Dane pogodowe - trzy zmienne',
    xaxis_title='Data',
    yaxis_title='Wartość',
    hovermode='x unified',
    height=600
)

fig.show()

### 3.3 Subplots

In [ ]:
from plotly.subplots import make_subplots

# Utwórz figurę z 2 wykresami
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Temperatura', 'Opady')
)

dane = df_pogoda.head(60)

# Wykres 1
fig.add_trace(
    go.Scatter(x=dane['data'], y=dane['temperatura'], mode='lines', name='Temperatura'),
    row=1, col=1
)

# Wykres 2
fig.add_trace(
    go.Bar(x=dane['data'], y=dane['opady'], name='Opady'),
    row=1, col=2
)

fig.update_layout(title_text='Dashboard pogodowy', height=500)
fig.show()

### 3.4 Wykresy 3D

In [ ]:
# 3D Scatter - INTERAKTYWNY!
fig = go.Figure(data=[go.Scatter3d(
    x=df_prac['wiek'],
    y=df_prac['staz'],
    z=df_prac['pensja'],
    mode='markers',
    marker=dict(
        size=8,
        color=df_prac['pensja'],    # Kolor według pensji
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Pensja')
    ),
    text=df_prac['imie']            # Hover pokazuje imię
)])

fig.update_layout(
    title='Wiek - Staż - Pensja (3D INTERAKTYWNY)',
    scene=dict(
        xaxis_title='Wiek',
        yaxis_title='Staż',
        zaxis_title='Pensja'
    ),
    height=700
)

fig.show()

print("\n💡 OBRACAJ WYKRES MYSZKĄ!")
print("- Kliknij i przeciągnij → obróć")
print("- Scroll → zoom")
print("- Najedź na punkt → zobacz szczegóły")

### 3.5 Wykres powierzchniowy 3D

In [ ]:
# Surface 3D - funkcja matematyczna
x = np.linspace(-5, 5, 50)
y = np.linspace(-5, 5, 50)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.sqrt(X**2 + Y**2))

fig = go.Figure(data=[go.Surface(
    x=X,
    y=Y,
    z=Z,
    colorscale='Viridis'
)])

fig.update_layout(
    title='Wykres powierzchniowy 3D (INTERAKTYWNY!)',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z = sin(√(x²+y²))'
    ),
    height=700
)

fig.show()

print("💡 Obracaj, zoomuj, eksploruj!")

---
## 4. Podstawy Dash - dashboardy webowe

**Dash** to framework do tworzenia interaktywnych dashboardów webowych.

**UWAGA:** Dash wymaga uruchomienia serwera. Kod poniżej pokazuje strukturę - uruchom go w osobnym skrypcie Python!

### Prosty przykład Dash

In [ ]:
# TEN KOD TO TYLKO PRZYKŁAD - NIE URUCHOMI SIĘ W NOTEBOOKU!
# Zapisz go jako osobny plik .py i uruchom: python dashboard.py

"""
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import pandas as pd

# Wczytaj dane
df = pd.read_csv('data/pracownicy.csv')

# Inicjalizuj aplikację
app = Dash(__name__)

# Layout aplikacji
app.layout = html.Div([
    html.H1('Dashboard Pracowników', style={'textAlign': 'center'}),
    
    html.Div([
        html.Label('Wybierz dział:'),
        dcc.Dropdown(
            id='dzial-dropdown',
            options=[{'label': d, 'value': d} for d in df['dzial'].unique()],
            value=df['dzial'].unique()[0],  # Domyślna wartość
            clearable=False
        )
    ], style={'width': '50%', 'margin': 'auto', 'padding': '20px'}),
    
    dcc.Graph(id='scatter-plot')
])

# Callback - aktualizuje wykres gdy zmieni się dropdown
@app.callback(
    Output('scatter-plot', 'figure'),
    Input('dzial-dropdown', 'value')
)
def update_graph(selected_dzial):
    # Filtruj dane
    filtered_df = df[df['dzial'] == selected_dzial]
    
    # Utwórz wykres
    fig = px.scatter(
        filtered_df,
        x='wiek',
        y='pensja',
        size='staz',
        hover_data=['imie'],
        title=f'Pracownicy działu: {selected_dzial}'
    )
    
    return fig

# Uruchom serwer
if __name__ == '__main__':
    app.run_server(debug=True)
"""

print("Zapisz powyższy kod jako 'dashboard.py' i uruchom: python dashboard.py")
print("Otworzy się http://127.0.0.1:8050/ w przeglądarce!")

### Struktura aplikacji Dash

Każda aplikacja Dash składa się z:

1. **Layout** - struktura HTML/komponenty
   - `html.Div`, `html.H1`, `html.P` - elementy HTML
   - `dcc.Graph` - wykres Plotly
   - `dcc.Dropdown`, `dcc.Slider`, `dcc.Input` - kontrolki

2. **Callbacks** - funkcje reagujące na zmiany
   - `@app.callback` - dekorator
   - `Input` - co wywołuje callback (np. zmiana dropdown)
   - `Output` - co callback aktualizuje (np. wykres)

### Komponenty Dash:

```python
# HTML
html.H1('Tytuł')
html.Div([...])
html.P('Paragraf')

# Core Components
dcc.Graph(id='wykres', figure=fig)
dcc.Dropdown(id='dropdown', options=[...], value=...)
dcc.Slider(id='slider', min=0, max=100, value=50)
dcc.Input(id='input', type='text', value='')
dcc.DatePickerRange(...)
```

### Praktyczny przykład - kod do uruchomienia

In [ ]:
# Zapisz to jako 'app_pogoda.py'

kod_dashboardu = '''
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import pandas as pd

# Dane
df_pogoda = pd.read_csv('data/pogoda.csv')
df_pogoda['data'] = pd.to_datetime(df_pogoda['data'])
df_pogoda['miesiac'] = df_pogoda['data'].dt.month

# App
app = Dash(__name__)

app.layout = html.Div([
    html.H1('Dashboard Pogodowy 🌤️', style={'textAlign': 'center', 'color': '#2c3e50'}),
    
    html.Div([
        html.Div([
            html.Label('Wybierz zmienną:', style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='zmienna',
                options=[
                    {'label': 'Temperatura', 'value': 'temperatura'},
                    {'label': 'Opady', 'value': 'opady'},
                    {'label': 'Wilgotność', 'value': 'wilgotnosc'}
                ],
                value='temperatura'
            )
        ], style={'width': '30%', 'display': 'inline-block'}),
        
        html.Div([
            html.Label('Miesiące:', style={'fontWeight': 'bold'}),
            dcc.RangeSlider(
                id='miesiac-slider',
                min=1, max=12, step=1,
                marks={i: str(i) for i in range(1, 13)},
                value=[1, 12]
            )
        ], style={'width': '60%', 'display': 'inline-block', 'marginLeft': '5%'})
    ], style={'padding': '20px'}),
    
    dcc.Graph(id='wykres-czasowy'),
    dcc.Graph(id='histogram')
])

@app.callback(
    [Output('wykres-czasowy', 'figure'),
     Output('histogram', 'figure')],
    [Input('zmienna', 'value'),
     Input('miesiac-slider', 'value')]
)
def update_graphs(zmienna, miesiace):
    # Filtruj dane
    filtered = df_pogoda[
        (df_pogoda['miesiac'] >= miesiace[0]) & 
        (df_pogoda['miesiac'] <= miesiace[1])
    ]
    
    # Wykres 1: Liniowy
    fig1 = px.line(filtered, x='data', y=zmienna, 
                   title=f'{zmienna.capitalize()} w czasie',
                   markers=True)
    
    # Wykres 2: Histogram
    fig2 = px.histogram(filtered, x=zmienna, nbins=20,
                        title=f'Rozkład: {zmienna}')
    
    return fig1, fig2

if __name__ == '__main__':
    app.run_server(debug=True)
'''

# Zapisz do pliku
with open('app_pogoda.py', 'w', encoding='utf-8') as f:
    f.write(kod_dashboardu)

print("✅ Plik 'app_pogoda.py' został utworzony!")
print("\nAby uruchomić dashboard:")
print("1. Otwórz terminal")
print("2. Uruchom: python app_pogoda.py")
print("3. Otwórz przeglądarkę: http://127.0.0.1:8050/")
print("\n💡 Dashboard będzie interaktywny - zmiana dropdown/slider → aktualizacja wykresów!")

---
## 4.1 Interaktywne slidery - kontrola parametrów wykresu

### Kiedy to się przydaje?

Wyobraź sobie, że analizujesz funkcję matematyczną z parametrem - np. `y = sin(k*x)`, gdzie `k` to częstotliwość.

Chcesz zobaczyć jak zmienia się wykres dla różnych wartości `k`. Zamiast tworzyć 10 osobnych wykresów, możesz stworzyć **JEDEN interaktywny** z sliderem!

**Slider** to suwak, który pozwala zmieniać parametr w czasie rzeczywistym - wykres automatycznie się aktualizuje.

### Jak to działa w Plotly?

Plotly pozwala dodać **slidery** i **przyciski** do wykresów Graph Objects. To bardziej zaawansowane niż Plotly Express, ale daje pełną kontrolę!

Zobaczmy praktyczny przykład:

In [ ]:
# === PRZYKŁAD: Slider kontrolujący częstotliwość funkcji sinus ===

import plotly.graph_objects as go
import numpy as np

# Generujemy dane dla różnych wartości k (częstotliwość)
x = np.linspace(0, 10, 100)

# Tworzymy figure
fig = go.Figure()

# Dodajemy krzywe dla różnych wartości k (1, 2, 3, 4, 5)
k_values = [1, 2, 3, 4, 5]

for k in k_values:
    y = np.sin(k * x)
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines',
        name=f'k = {k}',
        visible=(k == 1)  # Tylko pierwsza krzywa widoczna na starcie
    ))

# === SLIDER! ===
steps = []
for i, k in enumerate(k_values):
    # Dla każdej wartości k tworzymy "step" slidera
    step = dict(
        method="update",
        args=[{"visible": [j == i for j in range(len(k_values))]},  # Które trace widoczne
              {"title": f"Funkcja sin({k}·x)"}],  # Aktualizuj tytuł
        label=str(k)
    )
    steps.append(step)

sliders = [dict(
    active=0,
    currentvalue={"prefix": "Częstotliwość k = "},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(
    sliders=sliders,
    title="Funkcja sin(1·x)",
    xaxis_title="x",
    yaxis_title="y",
    height=500
)

fig.show()

print("\n💡 UŻYJ SLIDERA:")
print("  • Przesuń suwak → zmień częstotliwość k")
print("  • Wykres automatycznie się aktualizuje!")
print("\n✅ To jak animacja, ale kontrolujesz każdą klatkę!")

---
## 4.2 Interaktywne tabele - prezentacja danych tabelarycznych

### Kiedy to się przydaje?

Czasami wykres nie wystarczy - chcesz pokazać **dokładne liczby** w przejrzysty sposób.

**Przykłady:**
- Podsumowanie sprzedaży według regionów (liczby + procenty)
- Ranking produktów z wieloma metrykami
- Raport finansowy (przychody, koszty, zysk)

Plotly pozwala tworzyć **interaktywne tabele** z:
- Formatowaniem kolorów (np. zielony = dobry wynik, czerwony = zły)
- Sortowaniem
- Hover tooltips

### Jak tworzyć tabele w Plotly?

In [ ]:
# === PRZYKŁAD: Interaktywna tabela sprzedaży według kategorii ===

import plotly.graph_objects as go

# Dane: Podsumowanie sprzedaży
sprzedaz_kat = df_sprzedaz.groupby('kategoria').agg({
    'wartosc_sprzedazy': 'sum',
    'ilosc': 'sum'
}).reset_index()

sprzedaz_kat['srednia_cena'] = sprzedaz_kat['wartosc_sprzedazy'] / sprzedaz_kat['ilosc']
sprzedaz_kat = sprzedaz_kat.sort_values('wartosc_sprzedazy', ascending=False)

# Tabela w Plotly
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Kategoria</b>', '<b>Sprzedaż (PLN)</b>', '<b>Ilość</b>', '<b>Średnia cena</b>'],
        fill_color='paleturquoise',
        align='left',
        font=dict(size=14, color='black')
    ),
    cells=dict(
        values=[
            sprzedaz_kat['kategoria'],
            [f'{v:,.0f} PLN' for v in sprzedaz_kat['wartosc_sprzedazy']],
            sprzedaz_kat['ilosc'],
            [f'{v:,.2f} PLN' for v in sprzedaz_kat['srednia_cena']]
        ],
        fill_color='lavender',
        align='left',
        font=dict(size=12)
    )
)])

fig.update_layout(
    title='📊 Podsumowanie sprzedaży według kategorii',
    height=300
)

fig.show()

print("\n💡 ZASTOSOWANIA:")
print("  • Raporty dla management")
print("  • Prezentacje danych finansowych")
print("  • Dashboardy z metrykami")
print("\n✅ Czytelniej niż pandas DataFrame!")

---
## 4.3 Range selector - interaktywna selekcja zakresu czasu

### Kiedy to się przydaje?

Masz dane czasowe (np. temperatura przez rok, ceny akcji) i chcesz pozwolić użytkownikowi **łatwo wybierać okresy:**
- Ostatni miesiąc
- Ostatnie 3 miesiące
- Ostatnie 6 miesięcy
- Cały okres (All)

**Range selector** dodaje przyciski do szybkiego wyboru zakresu + slider do precyzyjnego dostrajania.

### Przykład: Dane pogodowe z selekcją zakresu

In [ ]:
# === PRZYKŁAD: Temperatura z range selector ===

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_pogoda['data'],
    y=df_pogoda['temperatura'],
    mode='lines',
    name='Temperatura',
    line=dict(color='red', width=2)
))

# === RANGE SELECTOR + SLIDER ===
fig.update_xaxes(
    rangeslider_visible=True,  # Dodaj slider na dole
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=3, label="3m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            dict(label="All", step="all")
        ])
    )
)

fig.update_layout(
    title='🌡️ Temperatura z selekcją zakresu',
    xaxis_title='Data',
    yaxis_title='Temperatura (°C)',
    height=600,
    hovermode='x unified'
)

fig.show()

print("\n💡 WYPRÓBUJ:")
print("  • Kliknij przycisk '1m', '3m', '6m' lub 'All'")
print("  • Użyj slidera na dole do precyzyjnej selekcji")
print("  • Zaznacz obszar na głównym wykresie → zoom")
print("\n✅ Idealne do analizy danych finansowych i time series!")

---
## 4.4 Multiple axes - dwie osie Y (różne skale)

### Kiedy to się przydaje?

Chcesz porównać dwie zmienne o **różnych skalach** na jednym wykresie:
- Temperatura (-5°C do 30°C) + Opady (0 do 50 mm)
- Sprzedaż (miliony PLN) + Liczba zamówień (setki)
- Cena akcji ($100-$200) + Wolumen (miliony sztuk)

Gdybyś użył jednej osi Y, jedna zmienna byłaby "zgubiona" na dole wykresu.

**Rozwiązanie:** Dual Y-axis - dwie osie Y, każda ze swoją skalą!

### Przykład: Temperatura i opady

In [ ]:
# === PRZYKŁAD: Dual Y-axis - temperatura i opady ===

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Tworzymy figure z secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Trace 1: Temperatura (primary y-axis)
fig.add_trace(
    go.Scatter(
        x=df_pogoda['data'],
        y=df_pogoda['temperatura'],
        name="Temperatura",
        line=dict(color='red', width=2)
    ),
    secondary_y=False  # Primary y-axis (lewa)
)

# Trace 2: Opady (secondary y-axis)
fig.add_trace(
    go.Bar(
        x=df_pogoda['data'],
        y=df_pogoda['opady'],
        name="Opady",
        marker=dict(color='blue'),
        opacity=0.5
    ),
    secondary_y=True  # Secondary y-axis (prawa)
)

# Ustawienia osi
fig.update_xaxes(title_text="Data")
fig.update_yaxes(title_text="<b>Temperatura (°C)</b>", secondary_y=False, title_font=dict(color='red'))
fig.update_yaxes(title_text="<b>Opady (mm)</b>", secondary_y=True, title_font=dict(color='blue'))

fig.update_layout(
    title='🌡️ Temperatura i opady - dual Y-axis',
    hovermode='x unified',
    height=500
)

fig.show()

print("\n💡 ZAUWAŻ:")
print("  • Lewa oś Y (czerwona) = Temperatura")
print("  • Prawa oś Y (niebieska) = Opady")
print("  • Każda ma swoją skalę!")
print("\n✅ Porównujesz zmienne różnej skali bez problemu!")
print("\n⚠️  UWAGA: Nie nadużywaj - może być mylące dla odbiorcy!")

---
## 4.5 Filled area charts - wypełnione wykresy obszarowe

### Kiedy to się przydaje?

**Area charts** (wykresy obszarowe) pokazują **zakres wartości** lub **przedział ufności:**
- Min-Max temperatura w ciągu dnia
- Przedział ufności prognozy (np. 95% confidence interval)
- Zakres zmienności cen (low-high)

Zamiast dwóch linii (min i max), wypełniasz obszar MIĘDZY nimi - łatwiej zobaczyć zakres!

### Typy wypełnienia:
- `fill='tozeroy'` - wypełnia do osi Y=0
- `fill='tonexty'` - wypełnia do poprzedniego trace (zakres min-max)
- `fill='toself'` - wypełnia zamknięty kształt

### Przykład: Zakres temperatur (min-max)

In [ ]:
# === PRZYKŁAD: Filled area chart - zakres temperatur ===

import plotly.graph_objects as go
import numpy as np

# Symulujemy dane: średnia temperatura + odchylenie
np.random.seed(42)
dni = pd.date_range('2025-01-01', periods=60, freq='D')
temp_avg = 5 + 10 * np.sin(np.linspace(0, 2*np.pi, 60)) + np.random.randn(60) * 2
temp_min = temp_avg - 5  # Min = średnia - 5°C
temp_max = temp_avg + 5  # Max = średnia + 5°C

fig = go.Figure()

# Trace 1: Linia górna (max)
fig.add_trace(go.Scatter(
    x=dni,
    y=temp_max,
    mode='lines',
    name='Maks',
    line=dict(width=0.5, color='rgba(255, 0, 0, 0.3)'),
    showlegend=False
))

# Trace 2: Linia dolna (min) + WYPEŁNIENIE
fig.add_trace(go.Scatter(
    x=dni,
    y=temp_min,
    mode='lines',
    name='Zakres temp',
    line=dict(width=0.5, color='rgba(255, 0, 0, 0.3)'),
    fill='tonexty',  # ⭐ Wypełnia DO poprzedniego trace!
    fillcolor='rgba(255, 0, 0, 0.2)'
))

# Trace 3: Średnia (linia środkowa)
fig.add_trace(go.Scatter(
    x=dni,
    y=temp_avg,
    mode='lines',
    name='Średnia',
    line=dict(width=2, color='red')
))

fig.update_layout(
    title='🌡️ Zakres temperatur (min-średnia-max)',
    xaxis_title='Data',
    yaxis_title='Temperatura (°C)',
    hovermode='x unified',
    height=500
)

fig.show()

print("\n💡 ZASTOSOWANIA:")
print("  • Przedziały ufności w prognozach")
print("  • Zakres zmienności (low-high) na giełdzie")
print("  • Min-Max wartości w czasie")
print("\n✅ Lepiej pokazuje niepewność niż dwie osobne linie!")

---
## 5. Inne przydatne narzędzia wizualizacji

### 5.1 Bokeh - interaktywne wykresy dla dużych danych

**Bokeh** to alternatywa dla Plotly, szczególnie dobra do:
- **Wielkich zbiorów danych** (miliony punktów)
- **Real-time streaming** (aktualizacja na żywo)
- **Server-side rendering** (renderowanie po stronie serwera)

#### Prosty przykład Bokeh:

```python
from bokeh.plotting import figure, show, output_notebook
import numpy as np

output_notebook()  # Wyświetlanie w notebooku

# Dane
x = np.linspace(0, 4*np.pi, 100)
y = np.sin(x)

# Wykres
p = figure(title="Funkcja sinus - Bokeh", width=800, height=400)
p.line(x, y, line_width=2, color='navy', alpha=0.8)
p.circle(x, y, size=5, color='red', alpha=0.5)

show(p)
```

**Kiedy Bokeh zamiast Plotly?**
- ✅ Masz >100k punktów danych (Bokeh lepiej to obsługuje)
- ✅ Potrzebujesz real-time updates (np. monitoring)
- ❌ Chcesz łatwość użycia → Plotly lepszy
- ❌ Potrzebujesz 3D → Plotly lepszy

---

### 5.2 Altair - gramatyka grafiki

**Altair** bazuje na Vega-Lite - deklaratywnej "gramatyce grafiki":
- Kod bardzo zwięzły i czytelny
- Dobrze integruje się z pandas
- Idealne do eksploracji danych

```python
import altair as alt

chart = alt.Chart(df_prac).mark_point().encode(
    x='wiek',
    y='pensja',
    color='dzial',
    size='staz'
)
chart.interactive()
```

---

### 5.3 Panel / HoloViews - zaawansowane dashboardy

**HoloViz ecosystem** (Panel + HoloViews + Param):
- Zaawansowane dashboardy naukowe
- Wsparcie dla wielkich danych (Datashader)
- Integracja z Jupyter, VS Code, standalone apps

**Kiedy używać:** Projekty badawcze, analiza dużych danych, GIS

---

### 5.4 Graph visualization - wizualizacja grafów i modeli ML

#### Przypadek użycia: Decision Trees (Drzewa decyzyjne)

Gdy trenujesz model Machine Learning (np. Decision Tree, Random Forest), chcesz **zobaczyć jak działa** - jakie decyzje podejmuje model.

**Narzędzia do wizualizacji grafów:**
- **Graphviz** - klasyczne narzędzie do grafów
- **dtreeviz** - specjalnie do decision trees
- **NetworkX + Matplotlib** - dla custom grafów

#### Przykład: Wizualizacja Decision Tree (sklearn)

```python
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.datasets import load_iris
import graphviz

# Załaduj dane iris
iris = load_iris()
X, y = iris.data, iris.target

# Wytrenuj model
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X, y)

# Wyeksportuj do Graphviz
dot_data = export_graphviz(
    clf,
    out_file=None,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True
)

# Renderuj
graph = graphviz.Source(dot_data)
graph.render("decision_tree")  # Zapisz do pliku decision_tree.pdf
# graph  # W notebooku wyświetli się bezpośrednio
```

**Rezultat:** Otrzymujesz **wizualną reprezentację** drzewa decyzyjnego - widzisz dokładnie jakie decyzje model podejmuje!

#### Inne zastosowania Graph Visualization:
- **Neural networks architecture** (warstwy sieci neuronowej)
- **Pipeline workflows** (kroki przetwarzania danych)
- **Dependency graphs** (zależności między pakietami)
- **Social networks** (sieci społecznościowe)

**Biblioteki:**
- `graphviz` - instalacja: `pip install graphviz` (wymaga zainstalowania Graphviz systemowo)
- `networkx` - grafy w Pythonie: `pip install networkx`
- `dtreeviz` - piękne drzewa: `pip install dtreeviz`

**Dokumentacja:**
- Graphviz: https://graphviz.org/
- NetworkX: https://networkx.org/
- sklearn trees: https://scikit-learn.org/stable/modules/tree.html

---

### Kiedy którego użyć?

| Narzędzie | Kiedy używać | Poziom trudności |
|-----------|-------------|------------------|
| **Plotly** | Dashboardy biznesowe, interaktywność, 3D | ⭐⭐ Średni |
| **Bokeh** | Wielkie zbiory danych, real-time streaming | ⭐⭐⭐ Zaawansowany |
| **Altair** | Eksploracja danych, prototypowanie | ⭐ Łatwy |
| **Panel** | Zaawansowane dashboardy naukowe | ⭐⭐⭐ Zaawansowany |
| **Graphviz** | Wizualizacja grafów, ML models | ⭐⭐ Średni |

**Rekomendacja dla początku:**
1. **Matplotlib** - podstawy, pełna kontrola
2. **Seaborn** - piękne wykresy statystyczne
3. **Plotly** - interaktywność i dashboardy
4. **Graphviz** - jeśli pracujesz z ML (opcjonalnie)

**Pamiętaj:** Nie musisz znać wszystkich! Opanuj dobrze 2-3 narzędzia i już jesteś ekspertem! 🚀

---
## Gratulacje! 🎉🎉🎉

# UKOŃCZYŁEŚ CAŁE SZKOLENIE!

## Co przeszedłeś:

### ✅ Moduł 1: Wstęp do wizualizacji
- Podstawy wizualizacji danych
- Dobre praktyki
- Przegląd narzędzi

### ✅ Moduł 2: Matplotlib (4 części!)
- Wszystkie typy wykresów
- Elementy składowe
- Stylowanie
- Wykresy 3D, subplots, zapisywanie

### ✅ Moduł 3: Seaborn (2 części)
- Wykresy relacji i rozkładów
- Wykresy kategoryczne
- Heatmapy i pairplot
- Stylowanie

### ✅ Moduł 4: Plotly & Dash
- Interaktywne wykresy
- Plotly Express i Graph Objects
- Wykresy 3D
- Podstawy Dash

## Umiejętności które zdobyłeś:

🎯 Tworzenie profesjonalnych wizualizacji  
🎯 Wybór odpowiedniego typu wykresu  
🎯 Stosowanie dobrych praktyk  
🎯 Analiza danych wizualnie  
🎯 Tworzenie dashboardów  
🎯 Interaktywne wizualizacje  

## Co dalej?

1. **Praktykuj!** - Wizualizuj własne dane
2. **Eksperymentuj** - Łącz różne biblioteki
3. **Dziel się** - Publikuj swoje wizualizacje
4. **Ucz się** - Śledź trendy w data viz

## Zasoby do dalszej nauki:

- **Matplotlib**: https://matplotlib.org/stable/gallery/index.html
- **Seaborn**: https://seaborn.pydata.org/examples/index.html
- **Plotly**: https://plotly.com/python/
- **Dash**: https://dash.plotly.com/

## Dziękuję za udział w szkoleniu!

**Miłej wizualizacji danych!** 📊📈📉